In [ ]:
%pyspark
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import DBSCAN

# Desactivamos Arrow por seguridad
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# ==========================================
# CONFIGURACIÓN DE NEGOCIO Y MODELOS
# ==========================================
# Pesos para el Hotspot Score
w_trafico     = 0.40  
w_permanencia = 0.30  
w_regulacion  = 0.30  

# Parámetros para descubrir Rutas (NLP DBSCAN)
nlp_eps = 0.4
nlp_min_samples = 2

# Parámetros para descubrir Hotspots (Spatial DBSCAN)
geo_eps_km = 2.5
geo_min_pings = 500

# ==========================================
# FASE 1: EXTRACCIÓN DE VIAJES Y SECUENCIAS
# ==========================================
df_journeys_base = spark.read.parquet("s3://datalake/charging_points/journeys/") \
    .select("userId", F.col("id").alias("journeyId"), F.col("route").alias("nombre_ruta_semantica"), "prevCentroid", "nextCentroid")

df_routes_prod = spark.read.parquet("s3://datalake/core/sql/table/route_daily/data/day=2025-12-10") \
    .select("userId", "journeyId", "nodes")

df_joined = df_routes_prod.join(df_journeys_base, ["userId", "journeyId"], "inner") \
    .filter(F.col("nodes").isNotNull() & (F.size("nodes") >= 2))

df_secuencias = df_joined.withColumn(
    "ruta_str", 
    F.expr("array_join(transform(nodes, x -> cast(x.id as string)), ' ')")
)

pdf_rutas = df_secuencias.groupBy("nombre_ruta_semantica", "ruta_str").agg(
    F.count("*").alias("num_viajes")
).filter(F.col("num_viajes") >= 1).toPandas()

if pdf_rutas.empty:
    raise ValueError("El DataFrame está vacío tras el filtro.")

# ==========================================
# FASE 2: CLUSTERING DE RUTAS (CÓDIGO 1 - NLP)
# ==========================================
resultados_mapping = []
resultados_agg_lineas = []

for nombre_ruta, pdf_grupo in pdf_rutas.groupby('nombre_ruta_semantica'):
    pdf_grupo = pdf_grupo.reset_index(drop=True)
    
    # Si la ruta es perfecta (1 sola variante), la guardamos sin DBSCAN
    if len(pdf_grupo) == 1:
        pdf_grupo['cluster_fisico_id'] = 0
        resultados_mapping.append(pdf_grupo)
        
        agg_negocio = pdf_grupo.copy()
        agg_negocio['trafico_cluster'] = agg_negocio['num_viajes']
        resultados_agg_lineas.append(agg_negocio)
        continue
        
    vectorizer = CountVectorizer(binary=True, lowercase=False, token_pattern=r"(?u)\S+")
    X_vect = vectorizer.fit_transform(pdf_grupo['ruta_str'])
    
    db_rutas = DBSCAN(eps=nlp_eps, min_samples=nlp_min_samples, metric='cosine')
    pdf_grupo['cluster_fisico_id'] = db_rutas.fit_predict(X_vect)
    
    # Filtramos el ruido (-1)
    rutas_validas = pdf_grupo[pdf_grupo['cluster_fisico_id'] != -1].copy()
    
    if not rutas_validas.empty:
        # Guardamos TODAS las variantes válidas para etiquetar camiones luego
        resultados_mapping.append(rutas_validas)
        
        # Agrupamos quedándonos con la variante más transitada para dibujar la línea
        rutas_validas = rutas_validas.sort_values('num_viajes', ascending=False)
        agg_negocio = rutas_validas.groupby('cluster_fisico_id').agg(
            trafico_cluster=('num_viajes', 'sum'),
            ruta_str=('ruta_str', 'first') 
        ).reset_index()
        
        agg_negocio['nombre_ruta_semantica'] = nombre_ruta
        resultados_agg_lineas.append(agg_negocio)

pdf_mapping_total = pd.concat(resultados_mapping, ignore_index=True)
pdf_agg_lineas = pd.concat(resultados_agg_lineas, ignore_index=True)

# ==========================================
# FASE 3: CONSTRUCCIÓN Y EXPORTACIÓN DE LÍNEAS (RUTAS)
# ==========================================
df_rutas_comerciales = spark.createDataFrame(pdf_agg_lineas)

df_nodos_ordenados = df_rutas_comerciales.withColumn("node_array", F.split(F.col("ruta_str"), " ")) \
    .select("nombre_ruta_semantica", "cluster_fisico_id", "trafico_cluster", F.posexplode("node_array").alias("orden", "node_id"))

df_graph_nodes = spark.read.parquet("s3://datalake/core/sql/table/routing_graph_nodes/data/") \
    .select(F.col("id").cast("string").alias("geo_node_id"), F.col("position.lat").alias("lat"), F.col("position.lon").alias("lon"))

df_rutas_geo = df_nodos_ordenados.join(df_graph_nodes, df_nodos_ordenados["node_id"] == df_graph_nodes["geo_node_id"], "inner").drop("geo_node_id")

df_rutas_geo_formateado = df_rutas_geo.withColumn("struct_coord", F.struct("orden", F.expr("format_string('%.4f %.4f', lon, lat)").alias("coord_par")))

df_kepler_lineas = df_rutas_geo_formateado.groupBy("nombre_ruta_semantica", "cluster_fisico_id", "trafico_cluster") \
    .agg(F.array_sort(F.collect_list("struct_coord")).alias("sorted_structs")) \
    .withColumn("lista_coords", F.expr("transform(sorted_structs, x -> x.coord_par)")) \
    .withColumn("geometry", F.concat(F.lit("LINESTRING("), F.array_join(F.col("lista_coords"), ", "), F.lit(")"))) \
    .select("nombre_ruta_semantica", "cluster_fisico_id", "trafico_cluster", "geometry")

ruta_export_lineas = "s3://datalake/charging_points/rutas_cluster___minSamples2/"
df_kepler_lineas.coalesce(1).write.csv(ruta_export_lineas, header=True, mode="overwrite", escape='"')


# ==========================================
# FASE 4: EL PUENTE (ETIQUETADO DE CAMIONES)
# ==========================================
# Cruzamos el mapeo validado con los viajes originales. ¡Los camiones "ruido" desaparecen aquí!
df_mapping_rutas = spark.createDataFrame(pdf_mapping_total[['nombre_ruta_semantica', 'ruta_str', 'cluster_fisico_id']])
df_viajes_etiquetados = df_secuencias.join(F.broadcast(df_mapping_rutas), ["nombre_ruta_semantica", "ruta_str"], "inner")


# ==========================================
# FASE 5: EXTRACCIÓN GEOGRÁFICA DE HOTSPOTS (CÓDIGO 2)
# ==========================================
df_nodos_time = df_viajes_etiquetados.select(
    "journeyId", "nombre_ruta_semantica", "cluster_fisico_id", "prevCentroid", "nextCentroid",
    F.explode("nodes").alias("node")
).select(
    "journeyId", "nombre_ruta_semantica", "cluster_fisico_id", "prevCentroid", "nextCentroid",
    F.col("node.id").cast("string").alias("node_id"), # Cast string para join seguro
    F.col("node.time").cast("long").alias("tiempo_relativo_seg")
)

# Peso por volumen de pings
df_node_traffic = df_nodos_time.groupBy("node_id").agg(F.count("*").alias("traffic_weight"))

# Join con coordenadas físicas
df_nodos_con_geo = df_node_traffic.join(
    df_graph_nodes.withColumnRenamed("geo_node_id", "node_id"), "node_id", "inner"
)
pdf_nodes_geo = df_nodos_con_geo.toPandas()

# Spatial DBSCAN
coords = np.radians(pdf_nodes_geo[['lat', 'lon']])
db_espacial = DBSCAN(eps=geo_eps_km/6371.0, min_samples=geo_min_pings, algorithm='ball_tree', metric='haversine')
pdf_nodes_geo['hotspot_id_raw'] = db_espacial.fit_predict(coords, sample_weight=pdf_nodes_geo['traffic_weight'])

valid_hotspots = pdf_nodes_geo[pdf_nodes_geo['hotspot_id_raw'] != -1].copy()
valid_hotspots['hotspot_id'] = "hs_" + valid_hotspots['hotspot_id_raw'].astype(str)

df_mapa_hotspots = spark.createDataFrame(valid_hotspots[['node_id', 'hotspot_id']])

# EPICENTRO: El nodo exacto con más tráfico del clúster
df_hotspot_coords = df_mapa_hotspots.join(df_nodos_con_geo, "node_id", "inner")
w_epicentro = Window.partitionBy("hotspot_id").orderBy(F.desc("traffic_weight"))

df_centroides_hotspots = df_hotspot_coords \
    .withColumn("rank", F.row_number().over(w_epicentro)) \
    .filter(F.col("rank") == 1) \
    .select("hotspot_id", F.col("lat").alias("latitude"), F.col("lon").alias("longitude"))


# ==========================================
# FASE 6: KPIs Y SCORING
# ==========================================
df_nodos_en_hotspots = df_nodos_time.join(df_mapa_hotspots, "node_id", "inner")

df_tiempos_completos = df_nodos_en_hotspots.groupBy(
    "journeyId", "nombre_ruta_semantica", "cluster_fisico_id", "hotspot_id", "prevCentroid", "nextCentroid"
).agg(
    F.min("tiempo_relativo_seg").alias("llegada_cluster_seg"),
    F.max("tiempo_relativo_seg").alias("salida_cluster_seg")
).withColumn(
    "permanencia_min", (F.col("salida_cluster_seg") - F.col("llegada_cluster_seg")) / 60
).withColumn(
    "conduccion_previa_h", F.col("llegada_cluster_seg") / 3600
)

df_hotspots_stats = df_tiempos_completos.groupBy("nombre_ruta_semantica", "cluster_fisico_id", "hotspot_id").agg(
    F.countDistinct("journeyId").alias("total_camiones"),
    F.round(F.avg("permanencia_min"), 1).alias("avg_permanencia_min"),
    F.round(F.avg("conduccion_previa_h"), 1).alias("avg_conduccion_horas")
)

df_hotspots_base = df_hotspots_stats.join(df_centroides_hotspots, "hotspot_id", "inner")

# TOP FROM / TO
df_od_texto = df_tiempos_completos.withColumn(
    "origen_str", F.concat_ws(",", F.round("prevCentroid.lat", 2), F.round("prevCentroid.lon", 2))
).withColumn(
    "destino_str", F.concat_ws(",", F.round("nextCentroid.lat", 2), F.round("nextCentroid.lon", 2))
)

def calcular_top_3_texto(df, campo_objetivo, nombre_columna_salida):
    w_rank = Window.partitionBy("nombre_ruta_semantica", "cluster_fisico_id", "hotspot_id").orderBy(F.desc("conteo_viajes"))
    return df.groupBy("nombre_ruta_semantica", "cluster_fisico_id", "hotspot_id", campo_objetivo).agg(F.count("*").alias("conteo_viajes")) \
        .withColumn("rank", F.row_number().over(w_rank)).filter(F.col("rank") <= 3) \
        .withColumn("linea_texto", F.concat(F.col(campo_objetivo), F.lit(" ("), F.col("conteo_viajes"), F.lit(" camiones)"))) \
        .groupBy("nombre_ruta_semantica", "cluster_fisico_id", "hotspot_id").agg(
            F.concat_ws(" | ", F.collect_list("linea_texto")).alias(nombre_columna_salida)
        )

df_hotspots_enriquecidos = df_hotspots_base \
    .join(calcular_top_3_texto(df_od_texto, "origen_str", "top_3_origenes"), ["nombre_ruta_semantica", "cluster_fisico_id", "hotspot_id"], "left") \
    .join(calcular_top_3_texto(df_od_texto, "destino_str", "top_3_destinos"), ["nombre_ruta_semantica", "cluster_fisico_id", "hotspot_id"], "left")

w_global = Window.partitionBy()

df_kepler_puntos = df_hotspots_enriquecidos \
    .withColumn("max_trafico", F.max("total_camiones").over(w_global)) \
    .withColumn("min_trafico", F.min("total_camiones").over(w_global)) \
    .withColumn("t_score", (F.col("total_camiones") - F.col("min_trafico")) / (F.col("max_trafico") - F.col("min_trafico") + 1e-5)) \
    .withColumn("p_score", F.when(F.col("avg_permanencia_min") >= 45.0, 1.0).otherwise(F.col("avg_permanencia_min") / 45.0)) \
    .withColumn("r_score", F.when(F.col("avg_conduccion_horas") >= 4.5, 1.0).otherwise(F.col("avg_conduccion_horas") / 4.5)) \
    .withColumn(
        "opportunity_score",
        F.round((F.lit(w_trafico) * F.col("t_score")) + (F.lit(w_permanencia) * F.col("p_score")) + (F.lit(w_regulacion) * F.col("r_score")), 3)
    ) \
    .select(
        "nombre_ruta_semantica",
        "cluster_fisico_id",
        "hotspot_id",
        "latitude",
        "longitude",
        "total_camiones",
        "avg_permanencia_min",
        "avg_conduccion_horas",
        "opportunity_score",
        "top_3_origenes",
        "top_3_destinos"
    )

# ==========================================
# FASE 7: EXPORTACIÓN DE PUNTOS (HOTSPOTS)
# ==========================================
ruta_export_puntos = "s3://datalake/charging_points/hotspots_odm_04_minSamples500_centroideMax/"
df_kepler_puntos.coalesce(1).write.csv(ruta_export_puntos, header=True, mode="overwrite", escape='"')

print("¡Proceso Atómico Finalizado! Generados LineStrings y Hotspots vinculados.")
df_kepler_puntos.orderBy("nombre_ruta_semantica", F.col("opportunity_score").desc()).show(10, truncate=30)